# 03 — Shortage Risk Classification
**CMD:** `jupyter lab` → **Run All**; alternative command: `python -m src.prediction --train-classification`.

**Purpose:** compare Decision Tree, Random Forest and XGBoost using chronological training/validation/test dates. Select by validation macro-F1 and report test performance.

In [ ]:
# CMD / purpose: Run this cell first in each notebook.
# Makes imports work whether Jupyter starts from the project root or notebooks/.
from pathlib import Path
import os, sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise RuntimeError('Open Jupyter from the blood-shortage-ml project directory.')
sys.path.insert(0, str(ROOT))
print('Project root:', ROOT)

In [ ]:
# CMD / purpose: train all classifiers, save best_classifier.joblib and classification_metrics.csv.
from src.prediction import train_classification
from src.feature_engineering import load_processed
results = train_classification(load_processed())
display(results.round(3))

In [ ]:
# CMD / purpose: display the final held-out test confusion matrix of selected model.
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from src.prediction import CLASSIFIER_PATH, chronological_splits, LABEL_TO_ID, RISK_LABELS
from src.feature_engineering import FEATURE_COLUMNS
df = load_processed().dropna(subset=['next_risk','target_date'])
test = chronological_splits(df)['test']
bundle = joblib.load(CLASSIFIER_PATH)
y_true = test.next_risk.map(LABEL_TO_ID).astype(int)
y_pred = bundle['pipeline'].predict(test[FEATURE_COLUMNS])
print('Best model:', bundle['model_name'])
print(classification_report(y_true, y_pred, labels=[0,1,2], target_names=list(RISK_LABELS), zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_true,y_pred,labels=[0,1,2],display_labels=RISK_LABELS,cmap='Blues',values_format='d')
plt.title('Held-out chronological test weeks')
plt.tight_layout();plt.show()

**Interpretation:** Macro-F1 gives equal importance to HIGH, MEDIUM and LOW categories; HIGH-risk recall must also be reported because missing shortages matters. The threshold-based labels are research proxies, not clinical ground truth.